# Vignette of plotting the likelihood of pop size estimate
In this vignette you will learn how to estimate effective pop sizes from ROH in specific length bins. This model assumes as panmictic population, so think of this estimates as the size of the ancestry pool at the time depth of the co-ancestry of the ROH you are fitting. The model used is described in the Supplement of https://doi.org/10.1101/2020.06.01.126730 and using the same framework introduced in https://www.genetics.org/content/205/3/1335

@author: Harald Ringbauer

In [ ]:
import os as os
import pandas as pd
import numpy as np
import sys

## Set the path
You can set the path here to the path you want to work in (relative data loads will be done from there)

In [ ]:
### Fill in your own path here!
path = "/mnt/archgen/users/xiaowen/Kamenice/1024/HapROH/"  # The Path to Package Midway Cluster
os.chdir(path)  # Set the right Path (in line with Atom default)
print(f"Set path to: {os.getcwd()}") # Show the current working directory. Should be HAPSBURG/Notebooks/ParallelRuns

# Load the data for the MLE analysis
We can analyse any set of individuals (including sets of a single individual). One has to load the ROH of these (as a list of lists), and passes them on to the MLE object.

The example ROH data here is hapROH output from the `./callROH_vignette` for 8 Levant Neolithic Chalcolithic individuals, run on Eigenstrat pseudo-haploid aDNA data that can be downloaded from `https://reich.hms.harvard.edu/sites/reich.hms.harvard.edu/files/inline-files/Levant_ChL.tar.gz`. The code here can be easily adapted to the ROH output from any set of indivdiuals.

In [ ]:
sys.path.insert(0, "/mnt/archgen/users/hringbauer/git/hapROH/package")  ### Leipzig  Development Version Harald
from hapsburg.PackagesSupport.fit_ne import MLE_ROH_Ne, load_roh_vec

In [ ]:
df1 = pd.read_csv("/mnt/archgen/users/xiaowen/Kamenice/1024/HapROH/0925/KNC_0925_roh05_originalID.csv", sep='\t')
df2 = df1[df1["sum_roh>20"]<50] # Remove close-kin consanguineous individuals - 50 cm in total of very long ROH (>20cm) is the cutoff chosen here.
df3=df2[df2["pop"]=="DIA"]
iidsA = df3["iid"].values # Load list of all iids
df4=df2[df2["pop"]=="EIA"]
iidsB = df4["iid"].values # Load list of all iids
print(f"Loaded {len(iidsA)} IIDs")

In [ ]:
iidsA

### Load ROH to analyze
This step requires that the individuals to analyse have been run with hapROH. Check the `callROH_vignette`
It loads all ROH lengths - in the step below you can define which lengths to actually fit.

In [ ]:
roh_vec_A = load_roh_vec(iids=iidsA, base_path = "./output_compiled/", suffix="_roh_full.csv")
roh_vec_B = load_roh_vec(iids=iidsB, base_path = "./output_compiled/", suffix="_roh_full.csv")

# Use Maximum Likelihood to fit Ne
Having the roh_vec loaded, now we can fit Ne. We use a class implemented in hapROH for that purpose. This retuns a pandas dataframe, with `coef` giving the most likely estimate, and two columns for the lower and upper bound of the 95% CI intervall for the estimate.

Note: Depending on demography (how many generations are alive, skew of reproductive success), the ratio of effective to census size is often **0.1-0.3x**. So the census size can be 3-10x bigger than the effective size! Check the popgen literature for further details.

Important: By default the reported estimates are for 2Ne. You have to divide by 2 to get the estimates in Ne.

In [ ]:
%%time
output = True
min_len = 4 # Min ROH length in cM to fit
max_len = 20 # Max ROH length in cM to fit
#combined_data = np.concatenate([roh_vec_A, roh_vec_B])
mle = MLE_ROH_Ne(start_params=1000, endog=roh_vec_A,
                 min_len=min_len, max_len=max_len,
                 chr_lgts=[],      # lengths of Chromosomes to fit (in cM). If len 0, use default for 1240K
                 error_model=False, output=True)
fit = mle.fit_ll_profile()

mle.summary/2  # to get estimates in terms of Ne

In [ ]:
import numpy as np
from scipy.stats import chi2

# -----------------------------
# Step 1: Fit each group separately
# -----------------------------
mle_A = MLE_ROH_Ne(
    start_params=1000,
    endog=roh_vec_A,       # Data for group A
    min_len=4,
    max_len=20,
    chr_lgts=[],
    error_model=False
)
fit_A = mle_A.fit_ll_profile()
#LL_A = mle_A.llf()  # log-likelihood at MLE for group A
#LL_A = -105.5168  # LBA
LL_A = -587.3922  # DIA

mle_B = MLE_ROH_Ne(
    start_params=1000,
    endog=roh_vec_B,       # Data for group B
    min_len=4,
    max_len=20,
    chr_lgts=[],
    error_model=False
)
fit_B = mle_B.fit_ll_profile()
#LL_B = mle_B.llf  # log-likelihood at MLE for group B
LL_B = -483.6169 #EIA
print(f"Group A log-likelihood: {LL_A:.3f}")
print(f"Group B log-likelihood: {LL_B:.3f}")

# -----------------------------
# Step 2: Fit both groups together under shared Ne
# -----------------------------
combined_data = np.concatenate([roh_vec_A, roh_vec_B])

mle_combined = MLE_ROH_Ne(
    start_params=1000,
    endog=combined_data,   # Combined dataset
    min_len=4,
    max_len=20,
    chr_lgts=[],
    error_model=False
)
fit_combined = mle_combined.fit_ll_profile()
#LL_combined = mle_combined.llf  # log-likelihood at MLE for combined model
#LL_combined = -589.1337 # LBA+EIA
LL_combined = -1071.0091 # EIA+DIA
print(f"Combined log-likelihood: {LL_combined:.3f}")

# -----------------------------
# Step 3: Compute LR test statistic
# -----------------------------
LR = -2 * (LL_combined - (LL_A + LL_B))

# -----------------------------
# Step 4: Compute p-value
# -----------------------------
p_value = 1 - chi2.cdf(LR, df=1)

print(f"Likelihood Ratio Statistic (LR): {LR:.3f}")
print(f"P-value: {p_value:.4f}")

# -----------------------------
# Interpretation
# -----------------------------
if p_value < 0.05:
    print("Significant difference: Groups A and B have different Ne (p < 0.05).")
else:
    print("No significant difference: Cannot reject same Ne for both groups (p >= 0.05).")


Ne=5284 (2855-11543 95% CI), that's a value typical for relatively large populations. It is a value where little ROH, even in the 4-8 cM category, as expected.

### [Advanced Parameter]
The default values for the parameter search in fit_ll_profile() is ns = np.logspace(2,5,1000)
ns is an array that specficies which 2Ne values to try (and look for the maximumg and 95%CI)

One can update the ns array to other values. That is for instance useful if the upper CI or even estimate hits 2N=100,000, the default limit.

# Alternative Mode for CI Intervalls
In the above method 95% Confidence Intervalls are fitted via the likelihood profile (calculating the log likelihood for a large number of 2Ne, and looking for the interval 1.92 LL units down from the Maximum Likelihood.)

A simpler and quicker way is to use the curvature of the likelihood (the so called Fisher Information matrix). This works well for a lot of data and small pop sizes - as the likelihood is approxiamted well by this Gaussian fit. However, when the likelihood is "flat", it is better to use the above method.

In [ ]:
%%time
output = True
min_len = 4 # Min ROH length in cM to fit
max_len = 20 # Max ROH length in cM to fit

mle = MLE_ROH_Ne(start_params=1000, endog=roh_vec,
                 min_len=4, max_len=20,
                 chr_lgts=[],      # lengths of Chromosomes to fit (in cM). If len 0, use default for 1240K
                 error_model=False, output=False)
fit = mle.fit()
#summary = fit.summary()
mle.summary/2  # to get estimates in terms of Ne

The estimates agree, but note that the CI are different. The "symmetric" and local approximation capture the order of magnitude of uncertainty, but is problematic in this case.

# Further Application: Fitting IBD_X
You can also use this machinery to fit pop sizes from IBD on the X. You have to set the chromosome length to chr_lgts=[180.85 * 2/3,], and first convert the length of the IBD_X to the sex-averaged rate (2/3 * (length of IBD in female map units)

In [ ]:
ns = np.logspace(2,5,num=1000)
ns